# RQ2 — Metadata Quality & Openness

**Research question:** To what extent do different countries meet Europeana's quality standards (Europeana Quality Score), and how open/reusable are their cultural datasets (copyright licenses)?

This notebook is organized in three parts:

1. **Data collection** — pull facet data and reusability counts from the Europeana Search API for the candidate countries.
2. **Quality & openness summary** — build the RQ2 table (content/metadata tier quality + license openness).
3. **Visualization & findings** — charts and discussion of the results.


## 1. Data collection

We query the Europeana Search API twice per country:

- once with `facet` on `contentTier`, `metadataTier`, `RIGHTS`, `TYPE`, `proxy_dc_subject`, `DATA_PROVIDER` (facet counts only, `rows=0`);
- once per `reusability` value (`open`, `restricted`, `permission`), since reusability is a request parameter and not something captured in the facet dump above.

The API key is read from an environment variable (`EUROPEANA_API_KEY`) rather than hard-coded, so the notebook can be shared/committed safely. Get a free key at https://api.europeana.eu/.


In [1]:
import os
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt

API_KEY = os.environ.get("EUROPEANA_API_KEY", "illardfon")
BASE_URL = "https://api.europeana.eu/record/v2/search.json"

# Facets tied to each RQ (names ARE case-sensitive, verified against Europeana docs):
# - contentTier, metadataTier -> RQ2 (quality score)
# - RIGHTS                    -> RQ2 (rights statement, raw URI form)
# - TYPE                      -> RQ1 (type of art: IMAGE, TEXT, 3D, etc.)
# - proxy_dc_subject          -> RQ1 (subject keywords)
# - DATA_PROVIDER             -> sanity check on institution-level granularity
FACETS = "contentTier,metadataTier,RIGHTS,TYPE,proxy_dc_subject,DATA_PROVIDER"

CANDIDATE_COUNTRIES = ["Italy", "Germany", "France", "Spain", "Portugal", "Netherlands"]

# Reusability is its own top-level parameter (open/restricted/permission), not a
# facet value under RIGHTS -> this is the more direct field for the RQ2
# "how open/reusable is the data" sub-question.
REUSABILITY_VALUES = ["open", "restricted", "permission"]

RAW_FILE = "europeana_raw.json"

# Europeana's own dividing line: tier 1 = does not meet the quality criteria,
# tiers 2/3/4 = meets it. Same logic for metadata: A/B meet it, C/0 don't.
QUALITY_CONTENT_TIERS = {"2", "3", "4"}
QUALITY_METADATA_TIERS = {"A", "B", "C"}

In [2]:
def fetch_country_facets(country, api_key):
    """Fetch facet counts (contentTier, metadataTier, RIGHTS, TYPE, ...) for a country."""
    params = {
        "wskey": api_key,
        "query": f'COUNTRY:"{country}"',
        "rows": 0,  # we only care about facets/counts, not actual records
        "facet": FACETS,
        "profile": "facets",
        "f.DATA_PROVIDER.facet.limit": 500,
    }
    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    return r.json()


def fetch_reusability(country, api_key):
    """Counts per reusability status (open/restricted/permission) for a country."""
    counts = {}
    for value in REUSABILITY_VALUES:
        params = {
            "wskey": api_key,
            "query": f'COUNTRY:"{country}"',
            "reusability": value,
            "rows": 0,
        }
        r = requests.get(BASE_URL, params=params)
        r.raise_for_status()
        counts[value] = r.json().get("totalResults", 0)
    return counts


Run the collection step only if you have a live API key. This writes the raw facet
responses to `europeana_exploration_raw.json`, which the offline analysis below relies on. If the
file already exists (e.g. from a previous run) this step can be skipped.


In [3]:
if API_KEY != "YOUR_API_KEY_HERE":
    raw_results = {}
    reusability_results = {}

    for country in CANDIDATE_COUNTRIES:
        try:
            raw_results[country] = fetch_country_facets(country, API_KEY)
            reusability_results[country] = fetch_reusability(country, API_KEY)
            print(f"Fetched {country}: {raw_results[country].get('totalResults')} total items")
        except Exception as e:
            print(f"Error fetching {country}: {e}")

    with open(RAW_FILE, "w") as f:
        json.dump(raw_results, f, indent=2)
else:
    print("No API key set (EUROPEANA_API_KEY) — skipping live collection, "
          "reusing previously saved data instead.")

Fetched Italy: 1832376 total items
Fetched Germany: 8701240 total items
Fetched France: 4724898 total items
Fetched Spain: 6581724 total items
Fetched Portugal: 139858 total items
Fetched Netherlands: 9204845 total items


## 2. Quality & openness summary

Two components make up RQ2:

- **Quality**: share of items meeting Europeana's contentTier (2/3/4) and metadataTier (A/B) thresholds.
- **Openness**: share of items under open, restricted, or permission-required licenses.


In [4]:
def facet_dict(country_data, facet_name):
    for facet in country_data["facets"]:
        if facet["name"] == facet_name:
            return {f["label"]: f["count"] for f in facet["fields"]}
    return {}


def quality_summary(raw_file=RAW_FILE):
    with open(raw_file) as f:
        raw = json.load(f)

    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        ct = facet_dict(data, "contentTier")
        mt = facet_dict(data, "metadataTier")

        content_ok = sum(v for k, v in ct.items() if k in QUALITY_CONTENT_TIERS)
        metadata_ok = sum(v for k, v in mt.items() if k in QUALITY_METADATA_TIERS)

        rows.append({
            "country": country,
            "total_items": total,
            "pct_contentTier_2plus": round(100 * content_ok / total, 1),
            "pct_contentTier_0": round(100 * ct.get("0", 0) / total, 1),
            "pct_metadataTier_ABC": round(100 * metadata_ok / total, 1),
            "pct_metadataTier_0": round(100 * mt.get("0", 0) / total, 1),
        })

    return pd.DataFrame(rows).sort_values("pct_contentTier_2plus", ascending=False)

def contenttier_breakdown(raw_file=RAW_FILE):
    """
    Disaggregated content-tier shares (1/2/3/4/0) per country.

    quality_summary() collapses tiers 2+3+4 into one 'pct_contentTier_2plus'
    number. But the rights-openness requirement only kicks in at tier 3/4
    (tier 2 has no rights condition) — so this breakdown is what actually
    shows whether a country's tier-2+ score is coming from rights-agnostic
    tier 2, or from rights-gated tier 3/4.
    """
    with open(raw_file) as f:
        raw = json.load(f)

    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        ct = facet_dict(data, "contentTier")

        row = {"country": country, "total_items": total}
        for tier in ["0", "1", "2", "3", "4"]:
            row[f"pct_tier_{tier}"] = round(100 * ct.get(tier, 0) / total, 1)
        rows.append(row)

    return pd.DataFrame(rows).sort_values("pct_tier_4", ascending=False)


tier_breakdown_df = contenttier_breakdown()
tier_breakdown_df

def metadatatier_breakdown(raw_file=RAW_FILE):
    """
    Disaggregated metadata-tier shares (A/B/C/0) per country.

    quality_summary() collapses A/B/C into one 'pct_metadataTier_ABC' number.
    Since A/B/C is an ascending scale (C = strictest/highest requirements),
    this breakdown shows whether a country's ABC score comes mostly from the
    loosest tier (A) or the strictest (C) -- the same kind of composition
    question contenttier_breakdown() answers for content tier.
    """
    with open(raw_file) as f:
        raw = json.load(f)

    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        mt = facet_dict(data, "metadataTier")

        row = {"country": country, "total_items": total}
        for tier in ["0", "A", "B", "C"]:
            row[f"pct_metadataTier_{tier}"] = round(100 * mt.get(tier, 0) / total, 1)
        rows.append(row)

    return pd.DataFrame(rows).sort_values("pct_metadataTier_C", ascending=False)


metadata_breakdown_df = metadatatier_breakdown()
metadata_breakdown_df


def openness_summary(reusability_results):
    rows = []
    for country, counts in reusability_results.items():
        total = sum(counts.values())
        rows.append({
            "country": country,
            "pct_open": round(100 * counts["open"] / total, 1) if total else None,
            "pct_restricted": round(100 * counts["restricted"] / total, 1) if total else None,
            "pct_permission": round(100 * counts["permission"] / total, 1) if total else None,
        })
    return pd.DataFrame(rows)

To assess the quality of the records contributed by different countries, Europeana classifies each item according to two complementary quality frameworks: Metadata Tier and Content Tier. Together, these indicators evaluate how well an object is described and how useful its associated digital representation is for users.

The **Content Tier** measures content quality and resuability by taking into account not just the quality of the digital resources, but also the rights statements and licenses applied to them: the fewer copyright restrictions placed on digital objects, the higher their potential for reuse and the higher value to end users. The classification ranges from 0 to 4, with Europeana considering CT2-CT4 as meeting its minimum publishing quality requirements, while CT0 and CT1 do not. the aggregated metric **pct_contentTier_2plus** represents the proportion of records that satisfy these quality criteria. However only tiers 3 and 4 add a rights requirement on top of technical quality, meaning that they're only reachable if the object carries a rights statement that allows reuse, for this reason the function **contenttier_breakdown()** further decomposes the distribution by individual tier, which is what allows checking whether a country's tier-2+ score is coming from rights-agnostic tier 2 content, or from
rights-gated tier 3/4 content, and therefore whether openness should be expected to track it.

The **Metadata Tier** measures the completeness and richness of the descriptive metadata provided with each record scored across three criteria
(language tagging, enabling elements, and contextual class links), with the overall tier determined by the *lowest*-scoring criterion. Europeana's documentation formally defines three levels, A through C, each with increasing thresholds. A "0" value also appears in the live API data and in Europeana's own example datasets, functioning as the equivalent of contentTier's tier 1: records that don't meet even the minimum bar for tier C. Our **pct_metadataTier_AB** metric captures the two tiers Europeana defines as meeting its quality standard.

In [5]:
if API_KEY != "YOUR_API_KEY_HERE":
    quality_df = quality_summary()
    openness_df = openness_summary(reusability_results)
    rq2_df = quality_df.merge(openness_df, on="country")
    rq2_df.to_csv("rq2_quality_openness.csv", index=False)
else:
    # Fall back to the already-computed table from a previous run.
    rq2_df = pd.read_csv("rq2_quality_openness.csv")

rq2_df.sort_values("pct_contentTier_2plus", ascending=False)

,country,total_items,pct_contentTier_2plus,pct_contentTier_0,pct_metadataTier_ABC,pct_metadataTier_0,pct_open,pct_restricted,pct_permission
0,Netherlands,9204845,87.5,1.1,85.6,14.4,76.1,10.9,13.0
1,Germany,8701240,76.2,3.1,94.8,5.2,28.3,50.2,21.5
2,Spain,6581724,65.3,1.1,70.0,30.0,44.5,37.7,17.8
3,Portugal,139858,59.1,2.2,97.6,2.4,97.7,0.1,2.2
4,Italy,1832376,44.3,2.5,86.4,13.6,11.8,31.2,56.9
5,France,4724898,41.4,17.4,70.7,29.3,11.3,63.3,25.4


## 3. Visualization & findings

In [6]:
import plotly.graph_objects as go

order = rq2_df.sort_values("pct_contentTier_2plus", ascending=False)["country"]
plot_df = rq2_df.set_index("country").loc[order]

# --- Quality chart (grouped bars) ---
fig_quality = go.Figure()
fig_quality.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_contentTier_2plus"],
                              name="Content tier 2+", marker_color="#4C72B0"))
fig_quality.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_metadataTier_ABC"],
                              name="Metadata tier A/B/C", marker_color="#DD8452"))
fig_quality.update_layout(
    barmode="group",
    title="Quality: content & metadata tier compliance",
    yaxis_title="% of items",
    height=500, width=600,
)
fig_quality.show()

In [7]:
import plotly.graph_objects as go

order = rq2_df.sort_values("pct_open", ascending=False)["country"]
plot_df = rq2_df.set_index("country").loc[order]

fig_openness = go.Figure()
fig_openness.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_open"],
                               name="Open", marker_color="#31A354"))
fig_openness.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_restricted"],
                               name="Restricted", marker_color="#FDAE6B"))
fig_openness.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_permission"],
                               name="Permission", marker_color="#DE2D26"))
fig_openness.update_layout(
    barmode="stack",
    title="Openness: license/reusability breakdown (ordered by openness)",
    yaxis_title="% of items",
    height=500, width=700,
)
fig_openness.show()

In [8]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

content_colors = {
    "pct_tier_0": "#E8E8E8",
    "pct_tier_1": "#C6DBEF",
    "pct_tier_2": "#6BAED6",
    "pct_tier_3": "#2171B5",
    "pct_tier_4": "#08306B",
}
metadata_colors = {
    "pct_metadataTier_0": "#E8E8E8",
    "pct_metadataTier_A": "#FDD0A2",
    "pct_metadataTier_B": "#FD8D3C",
    "pct_metadataTier_C": "#A63603",
}

# Same country order on both subplots -- sort by a meaningful reference,
# e.g. content tier 4 share, so the row order tells its own story
order = tier_breakdown_df.sort_values("pct_tier_4", ascending=True)["country"]  # ascending so highest ends up at top in a horizontal bar
ct = tier_breakdown_df.set_index("country").loc[order]
mt = metadata_breakdown_df.set_index("country").loc[order]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Content tier composition", "Metadata tier composition"),
    shared_yaxes=True,
    horizontal_spacing=0.08,
)

for col, color in content_colors.items():
    tier_label = col.replace("pct_tier_", "Content – Tier ")
    fig.add_trace(
        go.Bar(y=ct.index, x=ct[col], orientation="h",
               name=tier_label, marker_color=color, legendgroup="content"),
        row=1, col=1
    )

for col, color in metadata_colors.items():
    tier_label = col.replace("pct_metadataTier_", "Metadata – Tier ")
    fig.add_trace(
        go.Bar(y=mt.index, x=mt[col], orientation="h",
               name=tier_label, marker_color=color, legendgroup="metadata"),
        row=1, col=2
    )

fig.update_xaxes(title_text="% of items", row=1, col=1)
fig.update_xaxes(title_text="% of items", row=1, col=2)

fig.update_layout(
    barmode="stack",
    height=500, width=1150,
    title_text="Content vs. metadata tier composition, by country",
    legend=dict(title="Tier", tracegroupgap=10),
)

fig.show()

### Findings

**Quality (Europeana Quality Score).**

* The **Netherlands** leads on content tier: 87.5% of items reach content tier 2+, but its metadata score (85.6% A/B/C) is not the highest in the group, meanign that a mature digitization pipeline (Naturalis, Rijksmuseum) probably translates into strong technical/rights quality more than into top-tier descriptive metadata. 
* **Portugal** and **Germany** actually lead on metadata (97.6% and 94.8% A/B/C respectively), while sitting far apart on content tier (59.1% vs. 76.2%), a first sign that these two quality dimensions move independently rather than together. 
* **Italy** and **France** sit at the bottom for content tier (44.3% and 41.4%), even though both reach comparatively high metadata scores (86.4% and 70.7%) — France in particular combines a middling metadata score with the highest share of content tier 0 in the sample (17.4%), pointing to weak *technical/rights* quality rather than weak *descriptive* quality.

**Openness (license/reusability).**

* **Portugal** (97.7% open) and the **Netherlands** (76.1% open) are the most open. * **Italy** is the extreme opposite: only 11.8% open items against a striking 56.9% requiring explicit permission, the highest "permission" share in the dataset. 
* **France** is also low on openness (11.3%), but with a different pattern: it leans "restricted" (63.3%) rather than "permission-required."

**Quality and openness are not correlated**

Germany combines the second-highest metadata score in the group (94.5% A/B/C) with medium-low openness (28.3%). Italy is an even starker case: 86.4% of its items meet the metadata quality bar, yet only 11.8% are openly licensed — a near-90-point gap between how well items are *described* and how freely they can be *reused*. Portugal is the mirror image: excellent metadata (97.5%) paired with the highest openness in the sample (97.7%). Since content tier is *partly* tied to rights openness at Europeana's higher tiers (tier 3/4 require a reusable rights statement), some content-tier/openness overlap is expected by design — but metadata tier has no such built-in link, so cases like Italy and Germany are genuine empirical findings: institutions can invest heavily in descriptive richness while keeping their licensing restrictive. This supports reading technical/descriptive quality and openness as reflecting different institutional logics — digitization capacity and cataloguing effort on one side, policy choices around intellectual property on the other (e.g. Italy's high "in copyright" share likely reflects a more protective museum/archive system).


**pct_metadataTier_ABC** is an aggregate that could hide where the mass sits. A country could hit a high ABC score almost entirely via tier A (the loosest bar) while another hits it mostly via tier C (the strictest) — those are very different quality profiles that the aggregate alone can't distinguish, exactly like the tier-2-vs-3/4 issue you caught for content tier.

!! Right now these are just national averages, but national averages can mean two different things:
* distributed pattern: many institutions in a country converge independently in similar quality/licensing practices and this can genuinly reflect a national policy 
* concentrated pattern: one or two huge providers (e.g. a national library dumping millions of low-tier newspaper scans) single-handedly drag the country average down, while most other institutions might actually score well.

In [9]:
def provider_concentration(raw_file=RAW_FILE, top_n=5):
    with open(raw_file) as f:
        raw = json.load(f)

    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        providers = facet_dict(data, "DATA_PROVIDER")
        if not providers:
            continue

        sorted_counts = sorted(providers.values(), reverse=True)
        top_provider_name = max(providers, key=providers.get)

        top1_share = sorted_counts[0] / total
        top_n_actual_share = sum(sorted_counts[:top_n]) / total  # only the top N, explicitly

        rows.append({
            "country": country,
            "num_providers": len(providers),
            "top_provider": top_provider_name,
            "pct_top1_provider": round(100 * top1_share, 1),
            f"pct_top{top_n}_providers": round(100 * top_n_actual_share, 1),
        })

    return pd.DataFrame(rows).sort_values("pct_top1_provider", ascending=False)


concentration_df = provider_concentration(top_n=5)
concentration_df

,country,num_providers,top_provider,pct_top1_provider,pct_top5_providers
2,France,50,National Library of France,63.5,91.3
5,Netherlands,104,Naturalis Biodiversity Center,50.0,73.2
4,Portugal,40,Institute for Tropical Scientific Research,47.0,89.8
3,Spain,275,Virtual Library of Historical Press,26.4,49.5
1,Germany,375,Bavarian State Library,25.0,56.8
0,Italy,158,Cinecittà - Luce,24.2,58.1


* **pct_top1_provider** — what pct of all items in the country come from the single biggest provider? 
* **pct_top5_providers** -  what pct of all items are accounted for by the top 5 providers combined? This tells you how "thick" the tail is: if top8_share is low (e.g. 30%), most of the country's items are spread across many small/medium providers beyond the visible top 8. If it's high (e.g. 90%), the top 5 basically are the country's collection.


In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Order countries by concentration so the grid tells a story (most concentrated first)
plot_df = concentration_df.sort_values("pct_top1_provider", ascending=False).reset_index(drop=True)

fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "treemap"}]*3, [{"type": "treemap"}]*3],
    subplot_titles=plot_df["country"].tolist(),
    horizontal_spacing=0.03,
    vertical_spacing=0.12,
)

segment_colors = ["#203464", "#4b74a0", "#679e91"]  # dark = most concentrated, light = most distributed

for idx, row in plot_df.iterrows():
    r = idx // 3 + 1
    c = idx % 3 + 1

    top1 = row["pct_top1_provider"]
    top5_minus_1 = row["pct_top5_providers"] - top1
    others = 100 - row["pct_top5_providers"]

    fig.add_trace(
        go.Treemap(
            labels=["Top 1", "Rank 2–5", "Other"],
            parents=["", "", ""],
            values=[top1, top5_minus_1, others],
            marker=dict(colors=segment_colors, line=dict(width=1, color="white")),
            texttemplate="%{label}<br>%{value:.0f}%",
            textfont=dict(size=13, color="white"),
            hovertemplate="%{label}: %{value:.1f}%<extra></extra>",
        ),
        row=r, col=c,
    )

# Manual shared legend via dummy bar traces (treemaps don't support a clean shared legend natively)
for label, color in zip(["Top 1 provider", "Rank 2–5", "Other providers"], segment_colors):
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=color, name=label, showlegend=True))

fig.update_layout(
    title=dict(text="Provider concentration by country", x=0.02, font=dict(size=20)),
    height=650, width=1050,
    legend=dict(orientation="h", yanchor="bottom", y=-0.08, x=0.3),
    margin=dict(t=90, b=60, l=20, r=20),
    paper_bgcolor="white",
    font=dict(family="Arial, sans-serif"),
)

# Style subplot titles (country names) consistently
for annotation in fig["layout"]["annotations"]:
    annotation["font"] = dict(size=15, color="#333333")

fig.show()

### Geographic distribution of top-20 providers

To map where digitization activity concentrates within each country, we take each 
country's top 20 providers by item count and attempt to resolve a city-level location 
for each, using two sources in sequence:

1. **Europeana's own Organization entity profile** (direct address/geo data, curated by Europeana).
2. **Wikidata**, as a fallback for providers without a usable Europeana geo — first via
   the Wikidata URI already cross-referenced in Europeana's entity record where available,
   otherwise via a name-based search.

Providers that resolve through neither path are handled separately below.

In [11]:
def top_n_providers(raw_file=RAW_FILE, n=20):
    with open(raw_file) as f:
        raw = json.load(f)

    rows = []
    for country, data in raw.items():
        providers = facet_dict(data, "DATA_PROVIDER")
        top_providers = sorted(providers.items(), key=lambda x: x[1], reverse=True)[:n]
        for provider, count in top_providers:
            rows.append({"country": country, "provider": provider, "count": count})

    return pd.DataFrame(rows)

top20_df = top_n_providers()
top20_df

,country,provider,count
0,Italy,Cinecittà - Luce,442536
1,Italy,Historical Archive of the Presidency of the Re...,212160
2,Italy,National Central Library of Rome,200231
3,Italy,Internet Culturale,109101
4,Italy,"Department of Life Sciences, University of Tri...",101323
...,...,...,...
115,Netherlands,Netherlands Institute for Military History,68648
116,Netherlands,Historic Center Limburg,62704
117,Netherlands,IMSLP/Petrucci Music Library,61552
118,Netherlands,City Archives Breda,60870


In [12]:
import requests
import time
import json

# STEP 2 — Look up each provider in Europeana's own Organization entity system

def search_organization_entity(name, api_key):
    url = "https://api.europeana.eu/entity/suggest"
    params = {"wskey": api_key, "text": name, "type": "organization"}
    r = requests.get(url, params=params, timeout=10)
    if r.status_code != 200:
        return None
    items = r.json().get("items", [])
    return items[0]["id"] if items else None  # take the top match


def fetch_organization_entity(entity_id, api_key):
    entity_num = entity_id.rstrip("/").split("/")[-1]
    url = f"https://api.europeana.eu/entity/organization/{entity_num}"
    params = {"wskey": api_key}
    r = requests.get(url, params=params, timeout=10)
    if r.status_code != 200:
        return None
    return r.json()


def get_provider_geo(name, api_key):
    entity_id = search_organization_entity(name, api_key)
    if not entity_id:
        return {"matched": False}

    entity = fetch_organization_entity(entity_id, api_key)
    if not entity:
        return {"matched": True, "entity_id": entity_id, "has_geo": False}

    address = entity.get("hasAddress", {})
    geo = address.get("hasGeo", {})
    wikidata_uri = next((s for s in entity.get("sameAs", []) if "wikidata.org" in s), None)

    return {
        "matched": True,
        "entity_id": entity_id,
        "has_geo": bool(geo),
        "locality": address.get("locality"),
        "latitude": float(geo["lat"]) if geo.get("lat") else None,
        "longitude": float(geo["long"]) if geo.get("long") else None,
        "wikidata_uri": wikidata_uri,
    }


def enrich_providers_with_geo(top20_df, api_key):
    results = []
    for _, row in top20_df.iterrows():
        geo_info = get_provider_geo(row["provider"], api_key)
        results.append({
            "country": row["country"],
            "provider": row["provider"],
            "count": row["count"],
            **geo_info,
        })
        time.sleep(0.2)
    return pd.DataFrame(results)


geo_df = enrich_providers_with_geo(top20_df, API_KEY)

In [13]:
geo_df.to_csv("check.csv", index=False)

In [14]:
no_geo_but_matched = geo_df[(geo_df["matched"]) & (~geo_df["has_geo"].fillna(False))]
unmatched = geo_df[~geo_df["matched"]]

print(f"Matched with geo: {geo_df['has_geo'].sum()}")
print(f"Matched, no geo: {len(no_geo_but_matched)}")
print(f"Unmatched entirely: {len(unmatched)}")
geo_df[~geo_df["has_geo"].fillna(False)][["country", "provider", "matched", "wikidata_uri"]]

Matched with geo: 74
Matched, no geo: 25
Unmatched entirely: 21


/var/folders/b7/28t5nzhj4ygfhg81sls998h80000gn/T/ipykernel_27319/2009175117.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  no_geo_but_matched = geo_df[(geo_df["matched"]) & (~geo_df["has_geo"].fillna(False))]
/var/folders/b7/28t5nzhj4ygfhg81sls998h80000gn/T/ipykernel_27319/2009175117.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  geo_df[~geo_df["has_geo"].fillna(False)][["country", "provider", "matched", "wikidata_uri"]]


,country,provider,matched,wikidata_uri
0,Italy,Cinecittà - Luce,False,NaN
1,Italy,Historical Archive of the Presidency of the Re...,True,None
2,Italy,National Central Library of Rome,True,http://www.wikidata.org/entity/Q6382871
6,Italy,Cineteca di Bologna,False,NaN
7,Italy,Central Institute for the Union Catalogue of I...,True,None
8,Italy,Central Museum of the Risorgimento in Rome,True,None
9,Italy,Luigi Micheletti Foundation,True,http://www.wikidata.org/entity/Q25168251
10,Italy,Experimental Cinematography Center,True,None
11,Italy,Epigraphic Dabatase Bari,True,None
12,Italy,European Library of Information and Culture,True,None


### 46 remaining

In [15]:
from SPARQLWrapper import SPARQLWrapper, JSON
import re
import pandas as pd

HEADERS = {
    "User-Agent": "cultural-heritage-research/0.1 (university project; contact: your_email@studio.unibo.it)"}

def wikidata_search(name, limit=1):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "search": name,
        "language": "en",
        "format": "json",
        "limit": limit,
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=10)
    r.raise_for_status()
    return r.json().get("search", [])


def match_providers_to_wikidata(top20_df):
    matches = []
    for _, row in top20_df.iterrows():
        results = wikidata_search(row["provider"])
        if results:
            match = results[0]
            matches.append({
                "country": row["country"],
                "provider": row["provider"],
                "count": row["count"],
                "wikidata_id": match["id"],
                "wikidata_label": match.get("label"),
                "wikidata_description": match.get("description"),
            })
        else:
            matches.append({
                "country": row["country"],
                "provider": row["provider"],
                "count": row["count"],
                "wikidata_id": None,
                "wikidata_label": None,
                "wikidata_description": None,
            })
        time.sleep(0.2)  # be polite to the API
    return pd.DataFrame(matches)

def parse_point(coord_str):
    # Wikidata returns WKT format: "Point(lon lat)"
    match = re.match(r"Point\(([-\d.]+) ([-\d.]+)\)", coord_str)
    if match:
        lon, lat = match.groups()
        return float(lat), float(lon)
    return None, None

def fetch_coordinates(qids):
    """Given a list of Wikidata QIDs, return coordinates (P625) + city (P131)."""
    if not qids:
        return pd.DataFrame(columns=["wikidata_id", "wikidata_label", "city", "latitude", "longitude"])

    endpoint = "https://query.wikidata.org/sparql"
    sparql = SPARQLWrapper(endpoint, agent=HEADERS["User-Agent"])

    values_clause = " ".join(f"wd:{qid}" for qid in qids)
    query = f"""
    SELECT ?item ?itemLabel ?coord ?cityLabel WHERE {{
      VALUES ?item {{ {values_clause} }}
      ?item wdt:P625 ?coord .
      OPTIONAL {{ ?item wdt:P131 ?city . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    rows = []
    for r in results["results"]["bindings"]:
        lat, lon = parse_point(r["coord"]["value"])
        rows.append({
            "wikidata_id": r["item"]["value"].split("/")[-1],
            "wikidata_label": r["itemLabel"]["value"],
            "city": r.get("cityLabel", {}).get("value"),
            "latitude": lat,
            "longitude": lon,
        })
    return pd.DataFrame(rows)

def extract_qid(wikidata_uri):
    return wikidata_uri.rstrip("/").split("/")[-1]

In [16]:
# STEP 5 — Wikidata fallback #1: use the Wikidata URI Europeana already
# cross-referenced (no name-matching risk, since the URI came from
# Europeana's own curated record)

fallback_qids_direct = no_geo_but_matched["wikidata_uri"].dropna().apply(extract_qid).tolist()
fallback_coords_direct = fetch_coordinates(fallback_qids_direct)
fallback_coords_direct

,wikidata_id,wikidata_label,city,latitude,longitude
0,Q1526131,"KB, nationale bibliotheek",The Hague,52.081667,4.327500
1,Q6382871,Biblioteca Nazionale Centrale di Roma,Roma Capitale,41.906294,12.505950
2,Q25168251,Luigi Micheletti Foundation,Brescia,45.539306,10.213583


In [17]:
resolved_qids_step5 = set(fallback_coords_direct["wikidata_id"]) if len(fallback_coords_direct) else set()

In [18]:
still_unresolved = pd.concat([
    unmatched[["country", "provider", "count"]],
    no_geo_but_matched[~no_geo_but_matched["wikidata_uri"].apply(
        lambda u: pd.notna(u) and extract_qid(u) in resolved_qids_step5
    )][["country", "provider", "count"]],
], ignore_index=True)

print(f"Still unresolved before name search: {len(still_unresolved)}")

matched_fallback_df = match_providers_to_wikidata(still_unresolved)
print(matched_fallback_df["wikidata_id"].notna().sum(), "of", len(matched_fallback_df), "matched via name search")

fallback_qids_name = matched_fallback_df.loc[matched_fallback_df["wikidata_id"].notna(), "wikidata_id"].tolist()
fallback_coords_name = fetch_coordinates(fallback_qids_name)
fallback_coords_name

Still unresolved before name search: 43
14 of 43 matched via name search


,wikidata_id,wikidata_label,city,latitude,longitude
0,Q163255,Botanic Garden and Botanical Museum Berlin,Steglitz-Zehlendorf,52.455000,13.303600
1,Q655507,Deutsche Fotothek,Dresden,51.027810,13.736740
2,Q821048,Berlin-Brandenburg Economic Archive,Berlin,52.584017,13.316706
3,Q1092493,Cineteca di Bologna,Bologna,44.498866,11.336828
4,Q1954331,Museon-Omniversum,The Hague,52.088778,4.281000
5,Q1954426,Museum Catharijneconvent,Utrecht,52.087222,5.124167
6,Q3052794,Calouste Gulbenkian Foundation,Lisbon,38.737220,-9.154170
7,Q3378907,Philharmonie de Paris,Paris,48.891566,2.394070
8,Q3639582,Biblioteca Europea di Informazione e Cultura,Milan,45.472601,9.188543
9,Q41328625,Girona City Council,None,41.983138,2.824800


### STEP 7 — Consolidate everything into ONE final table

In [19]:
print(fallback_coords_direct.columns.tolist())

['wikidata_id', 'wikidata_label', 'city', 'latitude', 'longitude']


In [20]:
# 7a. Providers with direct Europeana geo
europeana_geo = geo_df[geo_df["has_geo"].fillna(False)][
    ["country", "provider", "count", "locality", "latitude", "longitude"]
].rename(columns={"locality": "city"})
europeana_geo["source"] = "europeana_entity"

# 7b. Providers resolved via Europeana's own Wikidata cross-reference
wikidata_direct = no_geo_but_matched.drop(columns=["latitude", "longitude"]).copy()
wikidata_direct["qid"] = wikidata_direct["wikidata_uri"].apply(
    lambda u: extract_qid(u) if pd.notna(u) else None
)
wikidata_direct = wikidata_direct.merge(
    fallback_coords_direct, left_on="qid", right_on="wikidata_id", how="inner"
)[["country", "provider", "count", "city", "latitude", "longitude"]]
wikidata_direct["source"] = "wikidata_via_europeana_link"

# 7c. Providers resolved via name-based Wikidata search
wikidata_named = matched_fallback_df.merge(
    fallback_coords_name, on="wikidata_id", how="inner"
)[["country", "provider", "count", "city", "latitude", "longitude"]]
wikidata_named["source"] = "wikidata_name_search"

provider_geo_final = pd.concat([europeana_geo, wikidata_direct, wikidata_named], ignore_index=True)
print(f"Total resolved: {len(provider_geo_final)} / {len(top20_df)}")
provider_geo_final

Total resolved: 88 / 120


/var/folders/b7/28t5nzhj4ygfhg81sls998h80000gn/T/ipykernel_27319/2018027808.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  europeana_geo = geo_df[geo_df["has_geo"].fillna(False)][


,country,provider,count,city,latitude,longitude,source
0,Italy,Internet Culturale,109101,Roma,41.906175,12.508847,europeana_entity
1,Italy,"Department of Life Sciences, University of Tri...",101323,Trieste,45.660689,13.795708,europeana_entity
2,Italy,Epigraphic Database Roma,86827,Rome,41.903763,12.514438,europeana_entity
3,Italy,Rossimoda Shoe Museum,13489,Vigonza,45.421946,11.976820,europeana_entity
4,Germany,Bavarian State Library,2176001,Munich,48.147493,11.580322,europeana_entity
...,...,...,...,...,...,...,...
83,Netherlands,Museon-Omniversum,93077,The Hague,52.088778,4.281000,wikidata_name_search
84,Netherlands,St. Catherine's Convent Museum,60795,Utrecht,52.087222,5.124167,wikidata_name_search
85,Italy,European Library of Information and Culture,35475,Milan,45.472601,9.188543,wikidata_name_search
86,Italy,Braidense National Library,21526,Milan,45.471947,9.187837,wikidata_name_search


### STEP 8 — What's left unresolved (for manual patching / documentation)

In [21]:
resolved_providers = set(provider_geo_final["provider"])
truly_unresolved = top20_df[~top20_df["provider"].isin(resolved_providers)]
truly_unresolved.sort_values("count", ascending=False)


,country,provider,count
60,Spain,Virtual Library of Historical Press,1735320
22,Germany,State Archives of Baden-Württemberg,827603
0,Italy,Cinecittà - Luce,442536
64,Spain,Galiciana. Arquivo Dixital de Galicia,234326
1,Italy,Historical Archive of the Presidency of the Re...,212160
26,Germany,Library of the Friedrich Ebert Foundation,201518
67,Spain,Digital Library of Andalusia,145620
68,Spain,Maresía: Prensa digitalizada y Patrimonio docu...,141919
30,Germany,Archives of Social Democracy,133032
31,Germany,State and University Library Hamburg Carl von ...,131101


In [22]:
# Manual lookup
manual_geo = pd.DataFrame([
    {"country": "Spain", "provider": "Virtual Library of Historical Press", "count": 1735320,
     "city": "Madrid", "latitude": 40.4206699298513, "longitude":  -3.696534590397076, "source": "manual"},
    {"country": "Germany", "provider": "State Archives of Baden-Württemberg", "count": 827603,
     "city": "Stuttgart", "latitude": 48.848397483733265, "longitude":  9.185433331037915, "source": "manual"},
    {"country": "Italy", "provider": "Cinecittà - Luce", "count": 442536,
     "city": "Rome", "latitude": 41.849810560200865, "longitude": 12.574519080840693, "source": "manual"},
    {"country": "Spain", "provider": "Galiciana. Arquivo Dixital de Galicia", "count": 234326,
     "city": "Santiago de Compostela", "latitude": 42.94991406654687, "longitude":  -8.551490373538044, "source": "manual"},
    {"country": "Italy", "provider": "Historical Archive of the Presidency of the Republic", "count": 212160,
     "city": "Rome", "latitude": 41.90046503888887, "longitude": 12.488921680843333, "source": "manual"},
    {"country": "Germany", "provider": "Library of the Friedrich Ebert Foundation", "count": 201518,
     "city": "Bonn", "latitude": 50.70257995350261, "longitude":  7.135440740863708, "source": "manual"},
    {"country": "Spain", "provider": "Digital Library of Andalusia", "count": 145620,
     "city": "Granada", "latitude": 37.18282776503285, "longitude": -3.6056596093053677, "source": "manual"},
    {"country": "Spain", "provider": "Maresía: Prensa digitalizada y Patrimonio documental", "count": 141919,
     "city": "San Cristóbal de La Laguna", "latitude": 28.46926099886812, "longitude": -16.30470874858524, "source": "manual"},
    {"country": "Germany", "provider": "Archives of Social Democracy", "count": 133032,
     "city": "Bonn", "latitude": 50.702368944477904, "longitude":7.134824767850492, "source": "manual"},
    {"country": "Germany", "provider": "State and University Library Hamburg Carl von Ossietzky", "count": 131101,
     "city": "Hamburg", "latitude": 53.56490744018377, "longitude": 9.985154223852446, "source": "manual"},
    {"country": "Spain", "provider": "Canary Islands Historical Photography Archive", "count": 121890,
     "city": "Las Palmas de Gran Canaria", "latitude": 28.10709632881129, "longitude": -15.417504652154406, "source": "manual"},
    {"country": "Germany", "provider": "Teßmann Library", "count": 115179,
     "city": "Italy", "latitude": 46.50323665221172, "longitude": 11.342929852259413, "source": "manual"},
    {"country": "Spain", "provider": "Centro de Estudios de Castilla - La Mancha", "count": 94223,
     "city": "Ciudad Real", "latitude": 38.993031000616305, "longitude": -3.9196156769745314, "source": "manual"},
    {"country": "Spain", "provider": "Foundation Virtual Library Miguel de Cervantes", "count": 70971,
     "city": "Alicante", "latitude": 38.38507646054709, "longitude": -0.5139165229076937, "source": "manual"},
    {"country": "Spain", "provider": "Digital Library of Madrid", "count": 69472,
     "city": "Madrid", "latitude": 40.39987943495303, "longitude": -3.690579871358554, "source": "manual"},
    {"country": "Italy", "provider": "Central Institute for the Union Catalogue of Italian Libraries", "count": 66672,
     "city": "Rome", "latitude": 41.90631839791796, "longitude": 12.50879375200811, "source": "manual"},
    {"country": "Netherlands", "provider": "IMSLP/Petrucci Music Library", "count": 61552,
     "city": None, "latitude": None, "longitude": None, "source": "manual"},
    {"country": "Italy", "provider": "Central Museum of the Risorgimento", "count": 53211,
     "city": "Rome", "latitude": 41.89413575264878, "longitude": 12.483798667349626, "source": "manual"},
    {"country": "France", "provider": "Palais Galliera - Musée de la Mode de la Ville de Paris", "count": 44495,
     "city": "Paris", "latitude": 48.86608033629242, "longitude": 2.2965618523973155, "source": "manual"},
    {"country": "Italy", "provider": "Experimental Cinematography Center", "count": 40538,
     "city": "Rome", "latitude": 41.851058155342265, "longitude": 12.569479552005237, "source": "manual"},
    {"country": "Italy", "provider": "Epigraphic Dabatase Bari", "count": 40283,
     "city": "Bari", "latitude": 41.12112736784277, "longitude": 16.868604802773277, "source": "manual"},
    {"country": "Italy", "provider": "Marciana National Library", "count": 29597,
     "city": "Venice", "latitude": 45.43353428986529, "longitude": 12.339422752198944, "source": "manual"},
    {"country": "Italy", "provider": "Turin Gallery for Modern and Contemporary Art", "count": 29395,
     "city": "Turin", "latitude": 45.065010771493675, "longitude": 7.66921379635602, "source": "manual"},
    {"country": "France", "provider": "National and University Library of Strasbourg", "count": 21320,
     "city": "Strasbourg", "latitude": 48.5872587063354, "longitude": 7.755901183065034, "source": "manual"},
    {"country": "Italy", "provider": "Library of the S. Pietro a Majella Conservatory", "count": 20154,
     "city": "Naples", "latitude": 40.849630767959596, "longitude": 14.252446682637897, "source": "manual"},
    {"country": "Italy", "provider": "Provincial Library Magna Capitana", "count": 13065,
     "city": "Foggia", "latitude": 41.45671868731526, "longitude": 15.558523953833472, "source": "manual"},
    {"country": "Italy", "provider": "Estense University Library", "count": 12372,
     "city": "Modena", "latitude": 44.64842579381791, "longitude": 10.920992367497412, "source": "manual"},
    {"country": "France", "provider": "Rhône-Alpes Laboratory for Historical Research", "count": 10046,
     "city": "Lyon", "latitude": 45.7334134679147, "longitude": 4.833417509886817, "source": "manual"},
    {"country": "Portugal", "provider": "MUDE – Museu do Design", "count": 1927,
     "city": "Lisbon", "latitude": 38.70924101252115, "longitude":  -9.136938317468571, "source": "manual"},
    {"country": "Portugal", "provider": "Fernando Pessoa's House", "count": 1210,
     "city": "Lisbon", "latitude": 38.716820396673036, "longitude": -9.162572105823484, "source": "manual"},
    {"country": "Portugal", "provider": "Lisbon's Film & Theatre School", "count": 774,
     "city": "Amadora", "latitude": 38.77728867945194, "longitude": -9.234926049524557, "source": "manual"},
    {"country": "Portugal", "provider": "Cinemateca Portuguesa - Museu do cinema", "count": 654,
     "city": "Lisbon", "latitude": 38.721224635535016, "longitude": -9.14865910212588, "source": "manual"},
])

manual_geo

provider_geo_final = pd.concat(
    [europeana_geo, wikidata_direct, wikidata_named, manual_geo],
    ignore_index=True
)

provider_geo_final
provider_geo_final.to_csv("map.csv", index=False)


One provider name ("Ministry of Culture," France) matched an incorrect entity during
automated lookup due to name ambiguity across countries (the query returned a Romanian
institution of the same generic name). This was identified through manual review and
corrected; given the risk of similarly generic institution names, all resolved
(country, city) pairs were spot-checked for geographic plausibility.

In [23]:
bad_row = provider_geo_final[
    (provider_geo_final["provider"].str.contains("Ministry of Culture", case=False)) &
    (provider_geo_final["country"] == "France")
]
bad_row

,country,provider,count,city,latitude,longitude,source
19,France,Ministry of Culture,173279,Bucharest,44.425230,26.110510,europeana_entity
21,France,"Ministry of Culture and Communication, Regiona...",110142,Strasbourg,48.587659,7.752661,europeana_entity


In [24]:
correct_coords = {"city": "Paris", "latitude": 48.86255840307406, "longitude": 2.3388283365026106}  # French Ministry of Culture, Paris 

mask = (provider_geo_final["provider"].str.contains("Ministry of Culture", case=False)) & \
       (provider_geo_final["country"] == "France")

provider_geo_final.loc[mask, ["city", "latitude", "longitude"]] = list(correct_coords.values())
provider_geo_final.loc[mask, "source"] = "manual correction (Europeana entity mismatched to wrong country)"

provider_geo_final[mask]

,country,provider,count,city,latitude,longitude,source
19,France,Ministry of Culture,173279,Paris,48.862558,2.338828,manual correction (Europeana entity mismatched...
21,France,"Ministry of Culture and Communication, Regiona...",110142,Paris,48.862558,2.338828,manual correction (Europeana entity mismatched...


In [107]:
import plotly.express as px
import numpy as np

provider_geo_final = provider_geo_final.sort_values("count", ascending=False)

min_size, max_size = 8, 40
c = provider_geo_final["count"]
provider_geo_final["size_mapped"] = min_size + (np.sqrt(c) - np.sqrt(c.min())) / (np.sqrt(c.max()) - np.sqrt(c.min())) * (max_size - min_size)

country_colors = {
    "Netherlands": "#a180ad", "Germany": "#E8B71D", "France": "#1f7f95",
    "Spain": "#bb521f", "Portugal": "#f4a64e", "Italy": "#90BE6D",
}

fig = px.scatter_map(
    provider_geo_final,
    lat="latitude",
    lon="longitude",
    color="country",
    size="size_mapped",
    hover_name="provider",
    hover_data={"city": True, "count": ":,", "size_mapped": False, "source": False,
                "latitude": False, "longitude": False},
    zoom=3.3,
    center=dict(lat=44, lon=-3),  # shifted west/south to keep islands + mainland both in frame
    map_style="carto-positron",   # light, clean basemap — no token needed
    title="Top 20 Providers by Location and Item Volume",
    color_discrete_map=country_colors,
)

fig.update_layout(
    height=650, width=950,
    legend_title="Country",
    margin=dict(l=10, r=10, t=60, b=10),
    title=dict(x=0.5,
        xanchor="center",
        font=dict(size=20, family="Roboto", color="#333")
        ),
    font=dict(family="Roboto", size=14, color="#444"),
)

def hex_to_rgba(hex_color, alpha=0.85):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r}, {g}, {b}, {alpha})"

def readable_font_color(hex_color):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    luminance = (0.299 * r + 0.587 * g + 0.114 * b) / 255
    return "#222222" if luminance > 0.6 else "#FBF9F5"

for trace in fig.data:
    country = trace.name
    base_color = country_colors.get(country, "#FBF9F5")
    trace.hoverlabel = dict(
        bgcolor=hex_to_rgba(base_color, 0.75),
        bordercolor=base_color,
        font=dict(color=readable_font_color(base_color)),
    )

fig.update_traces(
    marker=dict(opacity=0.75),
    hovertemplate=(
        "<b>%{hovertext}</b><br>" +
        "City: %{customdata[0]}<br>" +
        "Items in Europeana: %{customdata[1]:,}<br>" +
        "<extra></extra>"  # removes the trace-name box that appears by default
    )
)

fig.show()

In [106]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go   # <-- was missing

provider_geo_final = provider_geo_final.sort_values("count", ascending=False)

provider_geo_final["rank_in_country"] = (
    provider_geo_final.groupby("country")["count"]
    .rank(method="first", ascending=False)
)
provider_geo_final["is_top5"] = provider_geo_final["rank_in_country"] <= 5

min_size, max_size = 8, 40
c = provider_geo_final["count"]
provider_geo_final["size_mapped"] = min_size + (np.sqrt(c) - np.sqrt(c.min())) / (np.sqrt(c.max()) - np.sqrt(c.min())) * (max_size - min_size)

country_colors = {
    "Netherlands": "#a180ad", "Germany": "#E8B71D", "France": "#1f7f95",
    "Spain": "#bb521f", "Portugal": "#f4a64e", "Italy": "#90BE6D",
}

fig = px.scatter_map(
    provider_geo_final,
    lat="latitude",
    lon="longitude",
    color="country",
    size="size_mapped",
    size_max=max_size,     # <-- prevents Plotly re-scaling size_mapped again
    hover_name="provider",
    hover_data={"city": True, "count": ":,", "size_mapped": False, "source": False,
                "latitude": False, "longitude": False},
    zoom=3.3,
    center=dict(lat=44, lon=-3),
    map_style="carto-positron",
    title="Top 20 Providers by Location and Item Volume",
    color_discrete_map=country_colors,
)

n_all_traces = len(fig.data)  # one trace per country from the "all" view

# --- Style + hover for the BASE traces only (use selector, not a blanket update_traces) ---
fig.update_traces(
    selector=lambda t: t.name in country_colors,   # only the px-built base traces
    marker=dict(opacity=0.75),
    hovertemplate=(
        "<b>%{hovertext}</b><br>" +
        "City: %{customdata[0]}<br>" +
        "Items in Europeana: %{customdata[1]:,}<br>" +
        "<extra></extra>"
    ),
)

# --- Top-5-per-country traces (separate, added after) ---
top5 = provider_geo_final[provider_geo_final["is_top5"]]

for country in top5["country"].unique():
    sub = top5[top5["country"] == country]
    fig.add_trace(go.Scattermap(
        lat=sub["latitude"], lon=sub["longitude"],
        mode="markers+text",
        marker=dict(size=sub["size_mapped"], color=country_colors[country], opacity=0.9),
        text=sub["provider"],
        textposition="top right",
        textfont=dict(size=9, color="#222"),
        name=country,
        hovertext=sub["provider"],
        hovertemplate="<b>%{hovertext}</b><br>Items: %{customdata:,}<extra></extra>",
        customdata=sub["count"],
        visible=False,
        showlegend=False,
    ))

n_total = len(fig.data)
top5_indices = list(range(n_all_traces, n_total))

# --- Hover label colors, applied per trace by name (safe now, runs after all traces exist) ---
def hex_to_rgba(hex_color, alpha=0.85):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r}, {g}, {b}, {alpha})"

def readable_font_color(hex_color):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    luminance = (0.299 * r + 0.587 * g + 0.114 * b) / 255
    return "#222222" if luminance > 0.6 else "#FBF9F5"

for trace in fig.data:
    country = trace.name
    base_color = country_colors.get(country, "#FBF9F5")
    trace.hoverlabel = dict(
        bgcolor=hex_to_rgba(base_color, 0.75),
        bordercolor=base_color,
        font=dict(color=readable_font_color(base_color)),
    )

fig.update_layout(
    height=650, width=950,
    legend_title="Country",
    margin=dict(l=10, r=10, t=60, b=10),
    title=dict(x=0.5, xanchor="center", font=dict(size=20, family="Roboto", color="#333")),
    font=dict(family="Roboto", size=14, color="#444"),
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=0.5, y=1.08, xanchor="center",
            showactive=True,
            buttons=[
                dict(
                    label="Show all providers",
                    method="update",
                    args=[{"visible": [True] * n_all_traces + [False] * len(top5_indices)}],
                ),
                dict(
                    label="Top 5 per country",
                    method="update",
                    args=[{"visible": [False] * n_all_traces + [True] * len(top5_indices)}],
                ),
            ],
        )
    ],
)

fig.show()